# 02 — Target & feature distributions, outliers (team 07)

**Owner:** Lasya · **Branch:** `team-07-lasya`

**Questions to answer**
1. **Target distribution** — histogram of `TARGET` raw and on a log scale. Skewed? Zeros? Negative
   values? Two populations (like price-by-cabin in D2)?
2. **Outliers** — NumPy/IQR outlier counts for the target and the main numeric features. Are the
   outliers errors, or real (huge hub routes)? Real outliers stay — but they change the loss we pick.
3. **Key numeric features** — distributions of the numeric columns you expect to matter
   (`describe()`, histograms). Which need a log transform or scaling?
4. Does the target's spread differ a lot across groups (big vs small airports, route types)?

**Write the "why", not just the plot.** Every plot or number needs a sentence after it:
*"…therefore we should ___ when we model."* Those sentences go in the table at the bottom.

> **NDA reminder.** This is real American Airlines data. Never commit it, never paste rows or
> screenshots of it anywhere outside the team, and **clear outputs before every commit**
> (Kernel → Restart & Clear Output) so no data ends up in the notebook file.

In [ ]:
%run ./00_load_and_preprocess.ipynb
# gives you: pax, airports, TARGET, REPO_ROOT, RESULTS_DIR


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")

# Confirm that the preprocessing notebook loaded the data
print("Passenger dataset shape:", pax.shape)
print("Airport dataset shape:", airports.shape)
print("Target variable:", TARGET)

# Check that the target exists
assert TARGET in pax.columns, f"{TARGET} not found in pax"

# Identify numeric columns
numeric_cols = pax.select_dtypes(include=np.number).columns.tolist()

print("Number of numeric columns:", len(numeric_cols))


## 1. Target distribution

In [ ]:
# TODO: pax[TARGET].describe(); histogram raw + np.log1p(pax[TARGET])

In [ ]:
# Basic descriptive statistics
target_stats = pax[TARGET].describe()

display(target_stats)

# Check missing, zero, and negative values
print("Missing values:", pax[TARGET].isna().sum())
print("Zero values:", (pax[TARGET] == 0).sum())
print("Negative values:", (pax[TARGET] < 0).sum())

# Calculate skewness
print("Target skewness:", pax[TARGET].skew())



In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=pax,
    x=TARGET,
    bins=50,
    kde=True
)

plt.title("Distribution of TARGET (Raw)")
plt.xlabel(TARGET)
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# log1p requires values greater than or equal to -1.
# Passenger counts should normally be nonnegative.
if pax[TARGET].dropna().min() >= 0:

    target_log = np.log1p(pax[TARGET].dropna())

    plt.figure(figsize=(10, 5))

    sns.histplot(
        target_log,
        bins=50,
        kde=True
    )

    plt.title("Distribution of TARGET (Log1p)")
    plt.xlabel(f"log(1 + {TARGET})")
    plt.ylabel("Frequency")

    plt.tight_layout()
    plt.show()

else:
    print("Negative target values found. Investigate before applying log1p.")

*(Write here: what this shows and what it means for the model)*

## 2. Outliers (IQR)

In [ ]:
# TODO: q1, q3 = np.percentile(...); count values outside q1 - 1.5*IQR / q3 + 1.5*IQR for the target and numeric features

In [ ]:
def calculate_iqr_outliers(series):
    """
    Calculate IQR outlier statistics for a numeric pandas Series.
    Missing values are excluded.
    """

    values = series.dropna().to_numpy()

    if len(values) == 0:
        return {
            "Q1": np.nan,
            "Q3": np.nan,
            "IQR": np.nan,
            "Lower Bound": np.nan,
            "Upper Bound": np.nan,
            "Outlier Count": 0,
            "Outlier Percentage": np.nan
        }

    q1, q3 = np.percentile(values, [25, 75])

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = (
        (values < lower_bound) |
        (values > upper_bound)
    )

    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outliers.sum()),
        "Outlier Percentage": outliers.mean() * 100
    }


# Calculate outliers for all numeric columns
outlier_results = {}

for col in numeric_cols:
    outlier_results[col] = calculate_iqr_outliers(pax[col])

outlier_df = pd.DataFrame(outlier_results).T

display(
    outlier_df.round(2)
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    x=pax[TARGET].dropna()
)

plt.title(f"Boxplot of {TARGET}")
plt.xlabel(TARGET)

plt.tight_layout()
plt.show()

*(Write here: what this shows and what it means for the model)*

## 3. Numeric feature distributions

In [ ]:
# TODO: pax.describe(); histograms of the main numeric columns

In [ ]:
# Summary statistics for numeric features
numeric_summary = pax[numeric_cols].describe().T

# Add skewness
numeric_summary["skewness"] = pax[numeric_cols].skew()

# Add missing value counts
numeric_summary["missing_values"] = pax[numeric_cols].isna().sum()

display(numeric_summary.round(2))

In [ ]:
# Plot numeric features in groups of 4
features_to_plot = [
    col for col in numeric_cols
    if col != TARGET
]

for start in range(0, len(features_to_plot), 4):

    cols = features_to_plot[start:start + 4]

    pax[cols].hist(
        figsize=(14, 9),
        bins=30,
        layout=(2, 2),
        edgecolor="black"
    )

    plt.suptitle(
        "Numeric Feature Distributions",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()

In [ ]:
skewness = pax[numeric_cols].skew().sort_values(
    ascending=False
)

display(skewness.to_frame("Skewness"))

# Identify strongly skewed positive numeric features
log_candidates = []

for col in numeric_cols:

    values = pax[col].dropna()

    if len(values) == 0:
        continue

    if values.min() >= 0 and abs(values.skew()) > 1:
        log_candidates.append(col)

print("Potential log transformation candidates:")
print(log_candidates)

*(Write here: what this shows and what it means for the model)*

## 4. Spread by group

In [ ]:
# TODO: box plot of TARGET (log scale) by a grouping column

In [ ]:
# Show column names and data types
column_info = pd.DataFrame({
    "Column": pax.columns,
    "Data Type": pax.dtypes.astype(str).values,
    "Unique Values": pax.nunique(dropna=True).values
})

display(column_info)

In [ ]:
# CHANGE THIS to an actual grouping column in your dataset
GROUP_COL = "ROUTE_TYPE"

if GROUP_COL not in pax.columns:
    raise ValueError(
        f"{GROUP_COL} does not exist. Choose a column from pax.columns."
    )

# Create a temporary analysis dataframe
group_data = pax[[TARGET, GROUP_COL]].dropna().copy()

# Keep the most common 10 groups for readability
top_groups = (
    group_data[GROUP_COL]
    .value_counts()
    .head(10)
    .index
)

group_data = group_data[
    group_data[GROUP_COL].isin(top_groups)
].copy()

# Log-transform target for plotting
if (group_data[TARGET] < 0).any():
    raise ValueError("Negative target values need investigation.")

group_data["TARGET_LOG"] = np.log1p(
    group_data[TARGET]
)

plt.figure(figsize=(13, 6))

sns.boxplot(
    data=group_data,
    x=GROUP_COL,
    y="TARGET_LOG"
)

plt.title(f"Log Target Distribution by {GROUP_COL}")
plt.xlabel(GROUP_COL)
plt.ylabel(f"log(1 + {TARGET})")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

*(Write here: what this shows and what it means for the model)*

## Findings → modelling consequence (Lasya)

This table is what the team's feature set (05) is built from. Every row needs a **consequence**,
not just an observation — "*therefore we should ___ when we model.*"

| # | Observation (with the number) | Modelling consequence |
|---|---|---|
| 1 | *(Write here)* | *(Write here)* |
| 2 | *(Write here)* | *(Write here)* |
| 3 | *(Write here)* | *(Write here)* |